# 0) Setup

In [1]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index               # from budget.py
from humaidclf.batch import use_api_key_env           # context manager for key switching
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["test"]             # or ["train","dev","test"]
MODEL = "gpt-4o-mini"
RULES = RULES_1
TAG = "modeS-RULES1"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"

BATCH_TOKEN_LIMIT = 2_000_000  # Tier-1 cap
SAFETY_MARGIN = 0.90           # 10% headroom
MAX_OUTPUT_TOKENS = 40


# 1) Discover datasets (events/splits)

In [2]:
# --- discover datasets ---
def discover_tsvs(base: Path, splits: list[str]):
    items = []
    for event_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        event = event_dir.name
        for split in splits:
            tsv = event_dir / f"{event}_{split}.tsv"
            if tsv.exists():
                items.append({"event": event, "split": split, "tsv": str(tsv)})
    return pd.DataFrame(items)

df_sources = discover_tsvs(BASE, SPLITS)

# --- token budgeting ---
token_index = build_token_index(
    df_sources,
    model=MODEL,
    rules_text=RULES,
    batch_token_limit=BATCH_TOKEN_LIMIT,
    safety_margin=SAFETY_MARGIN,
    sample_size=200,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)

display(token_index)

df_fit     = token_index[token_index["fits_cap"]].reset_index(drop=True)
df_too_big = token_index[~token_index["fits_cap"]].reset_index(drop=True)

print("OK to run with Tier-1 key:")
display(df_fit[["event","split","num_rows","est_total_tokens","limit_used_%"]])

print("Too big for Tier-1 (use alternate key):")
display(df_too_big[["event","split","num_rows","est_total_tokens","limit_used_%"]])

,event,split,tsv,num_rows,avg_req_tokens,est_total_tokens,fits_cap,limit_used_%
8,kaikoura_earthquake_2016,test,Dataset\HumAID\kaikoura_earthquake_2016\kaikou...,435,441,191835,True,9.6
1,canada_wildfires_2016,test,Dataset\HumAID\canada_wildfires_2016\canada_wi...,445,439,195355,True,9.8
2,cyclone_idai_2019,test,Dataset\HumAID\cyclone_idai_2019\cyclone_idai_...,779,462,359898,True,18.0
4,hurricane_florence_2018,test,Dataset\HumAID\hurricane_florence_2018\hurrica...,1241,455,564655,True,28.2
7,hurricane_maria_2017,test,Dataset\HumAID\hurricane_maria_2017\hurricane_...,1442,441,635922,True,31.8
0,california_wildfires_2018,test,Dataset\HumAID\california_wildfires_2018\calif...,1461,450,657450,True,32.9
3,hurricane_dorian_2019,test,Dataset\HumAID\hurricane_dorian_2019\hurricane...,1508,457,689156,True,34.5
9,kerala_floods_2018,test,Dataset\HumAID\kerala_floods_2018\kerala_flood...,1582,462,730884,True,36.5
5,hurricane_harvey_2017,test,Dataset\HumAID\hurricane_harvey_2017\hurricane...,1805,440,794200,True,39.7
6,hurricane_irma_2017,test,Dataset\HumAID\hurricane_irma_2017\hurricane_i...,1862,440,819280,True,41.0


OK to run with Tier-1 key:


,event,split,num_rows,est_total_tokens,limit_used_%
0,kaikoura_earthquake_2016,test,435,191835,9.6
1,canada_wildfires_2016,test,445,195355,9.8
2,cyclone_idai_2019,test,779,359898,18.0
3,hurricane_florence_2018,test,1241,564655,28.2
4,hurricane_maria_2017,test,1442,635922,31.8
5,california_wildfires_2018,test,1461,657450,32.9
6,hurricane_dorian_2019,test,1508,689156,34.5
7,kerala_floods_2018,test,1582,730884,36.5
8,hurricane_harvey_2017,test,1805,794200,39.7
9,hurricane_irma_2017,test,1862,819280,41.0


Too big for Tier-1 (use alternate key):


,event,split,num_rows,est_total_tokens,limit_used_%


# 2) Run all datasets (sequentially)

In [3]:
# --- helpers to run a list of datasets ---
def run_list(dflist: pd.DataFrame, rules_text: str, model: str, tag: str):
    results = []
    for _, row in dflist.iterrows():
        event, split, tsv = row["event"], row["split"], row["tsv"]
        print(f"\n=== Running {event}/{split} ({model} | {tag}) ===")
        try:
            plan, preds, summary = run_experiment(
                dataset_path=tsv,
                rules=rules_text,
                model=model,
                tag=tag,
                dryrun_n=DRYRUN_N,
                poll_secs=POLL_SECS,
                out_root=OUT_ROOT,
                do_analysis=DO_ANALYSIS,
            )
            results.append({
                "event": event,
                "split": split,
                "run_dir": str(plan["dir"]),
                "predictions_csv": str(plan["predictions_csv"]),
                "macro_f1": summary.get("macro_f1"),
                "accuracy": summary.get("accuracy"),
                "num_total": summary.get("num_total_with_truth"),
            })
        except Exception as e:
            print(f"[ERROR] {event}/{split}: {e}")
            results.append({
                "event": event,
                "split": split,
                "run_dir": "ERROR",
                "predictions_csv": "",
                "macro_f1": float("nan"),
                "accuracy": float("nan"),
                "num_total": 0,
            })
    return pd.DataFrame(results)

# --- 1) Use OPENAI_API_KEY_1 for smaller datasets ---
with use_api_key_env("OPENAI_API_KEY_1"):
    print(">>> Using Tier-1 key (OPENAI_API_KEY_1)")
    df_runs_small = run_list(df_fit, RULES, MODEL, tag=f"{TAG}-TIER1")
    display(df_runs_small)

# --- 2) Use OPENAI_API_KEY_2 for larger datasets ---
if not df_too_big.empty:
    with use_api_key_env("OPENAI_API_KEY_2"):
        print(">>> Using alternate key (OPENAI_API_KEY_2)")
        df_runs_big = run_list(df_too_big, RULES, MODEL, tag=f"{TAG}-ALT")
        display(df_runs_big)
else:
    df_runs_big = pd.DataFrame()
    print("No large datasets; nothing to run with the alternate key.")

# (optional) save an index of what ran under which key
from datetime import datetime
idx_dir = Path(OUT_ROOT) / "_indexes"
idx_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")

df_runs_small.assign(key="OPENAI_API_KEY_1").to_csv(idx_dir / f"runs_tier1_{MODEL}_{TAG}_{stamp}.csv", index=False)
if not df_runs_big.empty:
    df_runs_big.assign(key="OPENAI_API_KEY_2").to_csv(idx_dir / f"runs_alt_{MODEL}_{TAG}_{stamp}.csv", index=False)
print("Saved run indexes in:", idx_dir)

>>> Using Tier-1 key (OPENAI_API_KEY_1)

=== Running kaikoura_earthquake_2016/test (gpt-4o-mini | modeS-RULES1-TIER1) ===
Macro-F1 (tiny sample): 0.7062770562770562
[batch batch_68fa7920632081908ba71af7304e044c] status = validating
[batch batch_68fa7920632081908ba71af7304e044c] status = in_progress
[batch batch_68fa7920632081908ba71af7304e044c] status = in_progress
[batch batch_68fa7920632081908ba71af7304e044c] status = in_progress
[batch batch_68fa7920632081908ba71af7304e044c] status = in_progress
[batch batch_68fa7920632081908ba71af7304e044c] status = in_progress
[batch batch_68fa7920632081908ba71af7304e044c] status = completed
Saved predictions to: runs\kaikoura_earthquake_2016\test\gpt-4o-mini\20251023-115106-modeS-RULES1-TIER1\predictions.csv
Macro-F1: 0.5850082680456262

=== Running canada_wildfires_2016/test (gpt-4o-mini | modeS-RULES1-TIER1) ===
Macro-F1 (tiny sample): 0.8277777777777778
[batch batch_68fa8047c8e48190ad9ffc6517e5eff8] status = validating
[batch batch_68fa8047c8e

,event,split,run_dir,predictions_csv,macro_f1,accuracy,num_total
0,kaikoura_earthquake_2016,test,runs\kaikoura_earthquake_2016\test\gpt-4o-mini...,runs\kaikoura_earthquake_2016\test\gpt-4o-mini...,0.585008,0.691954,435
1,canada_wildfires_2016,test,runs\canada_wildfires_2016\test\gpt-4o-mini\20...,runs\canada_wildfires_2016\test\gpt-4o-mini\20...,0.627784,0.757303,445
2,cyclone_idai_2019,test,runs\cyclone_idai_2019\test\gpt-4o-mini\202510...,runs\cyclone_idai_2019\test\gpt-4o-mini\202510...,0.590341,0.693196,779
3,hurricane_florence_2018,test,runs\hurricane_florence_2018\test\gpt-4o-mini\...,runs\hurricane_florence_2018\test\gpt-4o-mini\...,0.679533,0.747784,1241
4,hurricane_maria_2017,test,ERROR,,NaN,NaN,0
5,california_wildfires_2018,test,runs\california_wildfires_2018\test\gpt-4o-min...,runs\california_wildfires_2018\test\gpt-4o-min...,0.602359,0.703628,1461
6,hurricane_dorian_2019,test,runs\hurricane_dorian_2019\test\gpt-4o-mini\20...,runs\hurricane_dorian_2019\test\gpt-4o-mini\20...,0.521412,0.614058,1508
7,kerala_floods_2018,test,runs\kerala_floods_2018\test\gpt-4o-mini\20251...,runs\kerala_floods_2018\test\gpt-4o-mini\20251...,0.486286,0.675095,1582
8,hurricane_harvey_2017,test,runs\hurricane_harvey_2017\test\gpt-4o-mini\20...,runs\hurricane_harvey_2017\test\gpt-4o-mini\20...,0.553563,0.659280,1805
9,hurricane_irma_2017,test,runs\hurricane_irma_2017\test\gpt-4o-mini\2025...,runs\hurricane_irma_2017\test\gpt-4o-mini\2025...,0.545496,0.632116,1862


No large datasets; nothing to run with the alternate key.
Saved run indexes in: runs\_indexes


In [4]:
from humaidclf import run_experiment

plan, preds, summary = run_experiment(
    dataset_path="Dataset/HumAID/hurricane_maria_2017/hurricane_maria_2017_test.tsv",
    rules=RULES_1,
    model="gpt-4o-mini",
    tag="modeS-RULES1-TIER1",
    dryrun_n=20,
    poll_secs=300,
    do_analysis=True,
)
summary

Macro-F1 (tiny sample): 0.4450549450549451
[batch batch_68fd40ba7b408190845bc3cb317d4764] status = validating
[batch batch_68fd40ba7b408190845bc3cb317d4764] status = finalizing
[batch batch_68fd40ba7b408190845bc3cb317d4764] status = finalizing
[batch batch_68fd40ba7b408190845bc3cb317d4764] status = finalizing
[batch batch_68fd40ba7b408190845bc3cb317d4764] status = finalizing
[batch batch_68fd40ba7b408190845bc3cb317d4764] status = finalizing
[batch batch_68fd40ba7b408190845bc3cb317d4764] status = finalizing
[batch batch_68fd40ba7b408190845bc3cb317d4764] status = finalizing
[batch batch_68fd40ba7b408190845bc3cb317d4764] status = finalizing
[batch batch_68fd40ba7b408190845bc3cb317d4764] status = finalizing
[batch batch_68fd40ba7b408190845bc3cb317d4764] status = finalizing
[batch batch_68fd40ba7b408190845bc3cb317d4764] status = finalizing
[batch batch_68fd40ba7b408190845bc3cb317d4764] status = finalizing
[batch batch_68fd40ba7b408190845bc3cb317d4764] status = finalizing
[batch batch_68fd40

{'num_total_with_truth': 1441,
 'num_correct': 952,
 'num_incorrect': 489,
 'accuracy': 0.6606523247744622,
 'macro_f1': 0.5732628316952298,
 'labels': ['caution_and_advice',
  'displaced_people_and_evacuations',
  'infrastructure_and_utility_damage',
  'injured_or_dead_people',
  'missing_or_found_people',
  'requests_or_urgent_needs',
  'rescue_volunteering_or_donation_effort',
  'sympathy_and_support',
  'other_relevant_information',
  'not_humanitarian']}